# Gate 4a: zero-shot student rewrites (dev, 50 questions)

Rules: `docs/ROADMAP.md`, gate 4a. kvkk-rag's rewrite prompt unchanged, each model's chat template, bf16,
greedy, `max_new_tokens` 256, batch 1, dev set only. Retrieval is scored locally afterwards with
`scripts/eval_rewrites.py`.

Before running:
1. Runtime -> Change runtime type -> **L4 GPU**.
2. Secrets (key icon) -> `HF_TOKEN` = a Hugging Face read token of the account with Llama-3.1 access; notebook access on.
3. Runtime -> Run all. At the end a zip with the result files downloads; unzip it into the repo root.
4. Note the runtime minutes and compute units from Colab and add the row printed by the last cell to
   `results/colab_usage.csv`. Then disconnect and delete the runtime so it stops using compute units.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert "L4" in gpu, f"expected an L4 runtime (docs/ROADMAP.md, gate 4a), got {gpu!r}"

In [ ]:
import os
if not os.path.exists("/content/kvkk-rewriter"):
    subprocess.run(["git", "clone", "-q", "--recurse-submodules",
                    "https://github.com/EmirhanHasirci11/kvkk-rewriter.git", "/content/kvkk-rewriter"], check=True)
%cd /content/kvkk-rewriter
!git log -1 --format='%H %s' && git -C external/kvkk-rag log -1 --format='kvkk-rag %H'
!pip install -q "transformers>=4.51" accelerate

In [ ]:
import json, sys, time
from pathlib import Path

import torch, transformers
from google.colab import userdata

sys.path.insert(0, "scripts")
import student

TAG = "dev_v1"
HF_TOKEN = userdata.get("HF_TOKEN")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
dev = [json.loads(l) for l in Path("external/kvkk-rag/eval/golden.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
# warm-up question: the first training question (not from any eval set)
warmup = next(r["question"] for r in map(json.loads, Path("data/pairs_v1.jsonl").open(encoding="utf-8")) if r["split"] == "train")
out = {name: Path(f"results/rewrites_zs_{name}_{TAG}.jsonl") for name in student.MODELS}
for p in out.values():
    assert not p.exists(), f"{p} exists; results are never overwritten"
print(len(dev), "dev questions; torch", torch.__version__, "transformers", transformers.__version__)
print(student.SYSTEM)

In [ ]:
meta = {"gpu": gpu, "commit": COMMIT, "torch": torch.__version__, "transformers": transformers.__version__,
        "max_new_tokens": student.MAX_NEW_TOKENS, "models": {}}
t_start = time.time()
for name, hf_id in student.MODELS.items():
    print(f"\n=== {name} ({hf_id})")
    t0 = time.time()
    try:
        model, tok, stop = student.load(hf_id, token=HF_TOKEN)
    except Exception as e:    # e.g. no access to a gated model: record it, go on with the others
        print("LOAD FAILED:", repr(e)[:500])
        meta["models"][name] = {"hf_id": hf_id, "error": repr(e)[:500]}
        continue
    load_s = time.time() - t0
    torch.cuda.reset_peak_memory_stats()
    student.rewrite(model, tok, stop, warmup)
    rows = []
    for i, q in enumerate(dev, 1):
        r = student.rewrite(model, tok, stop, q["question"])
        rows.append({"id": q["id"], "question": q["question"], **r, "model": name, "hf_id": hf_id})
        print(f"[{i:2}/{len(dev)}] {r['latency_s']:.2f}s {r['new_tokens']:3} tok{' TRUNC' if r['truncated'] else ''}  {r['rewrite'][:110]}")
    out[name].write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
    lat = sorted(r["latency_s"] for r in rows)
    meta["models"][name] = {"hf_id": hf_id, "load_s": round(load_s, 1), "stop_ids": stop,
                            "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
                            "latency_p50_s": lat[len(lat) // 2], "latency_p95_s": lat[int(0.95 * (len(lat) - 1))],
                            "truncated": sum(r["truncated"] for r in rows)}
    print(meta["models"][name])
    del model
    torch.cuda.empty_cache()
meta["wall_min"] = round((time.time() - t_start) / 60, 1)
Path(f"results/colab_zs_{TAG}_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(json.dumps(meta, indent=2))

In [ ]:
import zipfile
from google.colab import files
paths = [p for p in out.values() if p.exists()] + [Path(f"results/colab_zs_{TAG}_meta.json")]
with zipfile.ZipFile(f"zs_{TAG}.zip", "w") as z:
    for p in paths:
        z.write(p)
files.download(f"zs_{TAG}.zip")
print("row for results/colab_usage.csv (fill in runtime_min and compute_units from Colab):")
print(f"{time.strftime('%Y-%m-%d')},notebooks/01_zeroshot.ipynb,{gpu},<runtime_min>,<compute_units>,zero-shot dev rewrites ({len(paths) - 1} models),commit {COMMIT[:7]}")